# Lesson 4.4 — Force / Velocity Duality
**Module 6 · Unit 4 · Lesson 16**

$\tau=J^\top F$. The force ellipsoid shares the velocity ellipsoid's axes but with **reciprocal** lengths $1/\sigma_i$: easy-to-move directions are weak-to-push directions.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]  # x,y rows for planar arms


## Velocity-ellipse axes σᵢ vs force-ellipse axes 1/σᵢ (same directions)

In [2]:
checks=[]
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
Jv=Jv_planar(P2,T2,np.array([0.5,0.8]))
Uv,Sv,Vv=np.linalg.svd(Jv)               # velocity ellipsoid: axes Uv, lengths Sv
JinvT=np.linalg.inv(Jv).T                # F = J^-T tau (square)
Uf,Sf,Vf=np.linalg.svd(JinvT)            # force ellipsoid: axes Uf, lengths Sf
print("velocity semi-axes (sigma):",np.round(Sv,3))
print("force    semi-axes        :",np.round(Sf,3))
print("reciprocal of velocity    :",np.round(1/Sv[::-1],3))
checks.append(np.allclose(np.sort(Sf),np.sort(1/Sv),atol=1e-9))

velocity semi-axes (sigma): [2.067 0.347]
force    semi-axes        : [2.882 0.484]
reciprocal of velocity    : [2.882 0.484]


## Same axis directions, reciprocal lengths -> easy-to-move = weak-to-push

In [3]:
# columns of Uv and Uf span the same lines (up to sign/order)
align=abs(Uv.T@Uf)   # ~ permutation/identity in magnitude
print("axis alignment |Uv^T Uf| (≈ permutation):\n",np.round(align,3))
checks.append(np.allclose(np.sort(align.flatten())[-2:],[1,1],atol=1e-6))

axis alignment |Uv^T Uf| (≈ permutation):
 [[0. 1.]
 [1. 0.]]


## Near a singularity: vanishing velocity axis -> diverging force axis

In [4]:
Jn=Jv_planar(P2,T2,np.array([0.4,0.05]))     # near straight
sv=np.linalg.svd(Jn,compute_uv=False)
print("velocity sigma_min:",round(sv[-1],4)," -> force semi-axis 1/sigma_min:",round(1/sv[-1],2))
checks.append(1/sv[-1]>5)
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

velocity sigma_min: 0.0224  -> force semi-axis 1/sigma_min: 44.73
All checks passed.
